# DB → LLM SQL → FHIR 단계별 테스트

이 노트북은 현재 프로젝트를 다음 순서로 점검합니다.

1. `.env`와 연결 설정
2. Demo Patient NDJSON에서 환자 key 추출
3. PostgreSQL 사용자 스키마 자동 탐색
4. SQL/FHIR Vector DB 및 RAG 검색
5. LLM SQL 생성·DB 조회·Self-Correction
6. 단일 환자 LLM FHIR 생성
7. FHIR Bundle 검증
8. 선택적 demo 환자 10명 배치 실행

비밀번호는 출력하지 않습니다. LLM 호출, Vector 재구축, 외부 Validator 및 배치 실행은 아래 플래그를 명시적으로 켜야 합니다.

In [1]:
from pathlib import Path
import json
import sys
from dataclasses import replace
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "main.py").is_file():
    candidates = [p for p in PROJECT_ROOT.parents if (p / "main.py").is_file()]
    if not candidates:
        raise RuntimeError("main.py가 있는 프로젝트 루트에서 노트북을 실행하세요.")
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 시간이 오래 걸리거나 외부 상태를 변경하는 단계는 필요할 때만 True로 바꿉니다.
RUN_VECTOR_REBUILD = True
RUN_RAG_SEARCH = True
RUN_LLM_SQL = True
RUN_SINGLE_PATIENT_PIPELINE = True
RUN_EXTERNAL_VALIDATOR = False
RUN_BATCH_10 = True

NOTEBOOK_OUTPUT = PROJECT_ROOT / "fhir_export_notebook_test"
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /home/jhkim/fhir_agent


## 1. `.env` 및 실행 설정 테스트

DB URL은 출력하지 않고 설정 로딩, 파일 경로, 모델과 Vector 테이블 이름만 확인합니다.

In [2]:
from fhir_agent_pipeline.config import load_settings

settings = load_settings()
config_summary = {
    "source_profile": str(settings.source_profile_path),
    "source_data_path": str(settings.source_data_path),
    "fhir_schema_path": str(settings.fhir_schema_path),
    "ollama_model": settings.ollama_model,
    "vectors_enabled": settings.vectors_enabled,
    "sql_vector_table": settings.sql_vector_table,
    "fhir_vector_table": settings.fhir_vector_table,
    "output_resources": settings.fhir_output_resources,
    "validator_configured": settings.fhir_validator_jar is not None,
}
assert settings.source_profile_path.is_file()
assert settings.fhir_schema_path.is_file()
display(config_summary)


{'source_profile': '/home/jhkim/fhir_agent/profiles/mimic_iv/profile.json',
 'source_data_path': '/home/mdhi/physionet.org/files/mimic-iv-fhir-demo',
 'fhir_schema_path': '/home/jhkim/fhir_agent/db/fhir.schema.json',
 'ollama_model': 'gemma4:e4b',
 'vectors_enabled': True,
 'sql_vector_table': 'source_sql_schema_vectors',
 'fhir_vector_table': 'fhir_schema_vectors',
 'output_resources': ['Patient',
  'Encounter',
  'Condition',
  'Procedure',
  'Observation'],
 'validator_configured': True}

## 2. Demo cohort 환자 key 추출 테스트

LLM은 환자 ID를 추론하지 않습니다. Source profile이 지정한 Patient NDJSON identifier에서 key를 읽습니다.

In [3]:
from fhir_agent_pipeline.export_service import (
    iter_cohort_patient_ids,
    resolve_cohort_patient_file,
)

cohort_file = resolve_cohort_patient_file(settings)
patient_keys = list(iter_cohort_patient_ids(settings, limit=10))
assert cohort_file.is_file()
assert len(patient_keys) == 10
assert len(set(patient_keys)) == 10
PATIENT_KEY = patient_keys[1]
display({"cohort_file": str(cohort_file), "patient_keys": patient_keys})


{'cohort_file': '/home/mdhi/physionet.org/files/mimic-iv-fhir-demo/2.1.0/fhir/MimicPatient.ndjson.gz',
 'patient_keys': [10007795,
  10007928,
  10009628,
  10011398,
  10004457,
  10003046,
  10014078,
  10003400,
  10020187,
  10038999]}

## 3. PostgreSQL 스키마 자동 탐색 테스트

`public`, `information_schema`, `pg_*`, `*derived*`를 제외한 사용자 스키마의 테이블·컬럼·PK·FK를 DB metadata에서 읽습니다.

In [4]:
from fhir_agent_pipeline.db import build_engine, SourceSchemaCatalog

engine = build_engine(settings)
try:
    catalog = SourceSchemaCatalog.load(settings, engine)
    assert catalog.tables
    assert all(t.schema.lower() not in {"public", "information_schema"} for t in catalog.tables)
    assert all(not t.schema.lower().startswith("pg_") for t in catalog.tables)
    schema_summary = {
        "schemas": sorted(catalog.schemas),
        "table_count": len(catalog.tables),
        "sample_tables": [t.qualified_name for t in catalog.tables[:10]],
    }
    display(schema_summary)
finally:
    engine.dispose()


{'schemas': ['mimiciv_derived', 'mimiciv_hosp', 'mimiciv_icu'],
 'table_count': 59,
 'sample_tables': ['mimiciv_derived.antibiotic',
  'mimiciv_derived.bg',
  'mimiciv_derived.chemistry',
  'mimiciv_derived.complete_blood_count',
  'mimiciv_derived.crrt',
  'mimiciv_derived.dobutamine',
  'mimiciv_derived.dopamine',
  'mimiciv_derived.enzyme',
  'mimiciv_derived.epinephrine',
  'mimiciv_derived.gcs']}

## 4. Vector DB 및 RAG 검색 테스트

기본 실행은 기존 Vector 테이블을 검사합니다. `RUN_VECTOR_REBUILD=True`이면 현재 DB metadata와 FHIR JSON Schema로 Vector DB를 갱신합니다.

In [5]:
from fhir_agent_pipeline.vector_store import (
    PgVectorKnowledgeStore,
    build_knowledge_bases,
)

if RUN_VECTOR_REBUILD:
    rebuild_result = build_knowledge_bases(settings)
    display(rebuild_result)

store = PgVectorKnowledgeStore(settings)
try:
    vector_reports = [
        store.inspect_table(settings.sql_vector_table),
        store.inspect_table(settings.fhir_vector_table),
    ]
    for report in vector_reports:
        assert report["exists"] and report["compatible"] and report["rows"] > 0
    display(vector_reports)

    if RUN_RAG_SEARCH:
        sql_matches = store.search(
            settings.sql_vector_table,
            "patient demographics encounter identifier and admission time",
            k=5,
            profile_scoped=True,
        )
        fhir_matches = store.search(
            settings.fhir_vector_table,
            "FHIR R4 Patient required properties references datatypes",
            k=5,
            profile_scoped=False,
        )
        assert sql_matches and fhir_matches
        display({
            "sql_rag": [
                {"metadata": x["metadata"], "similarity": round(float(x["similarity"]), 4)}
                for x in sql_matches
            ],
            "fhir_rag": [
                {"metadata": x["metadata"], "similarity": round(float(x["similarity"]), 4)}
                for x in fhir_matches
            ],
        })
finally:
    store.engine.dispose()


{'fhir_documents': {'total': 11499, 'embedded': 0, 'reused': 11499},
 'sql_documents': {'total': 59, 'embedded': 0, 'reused': 59},
 'pruned_documents': {'fhir': 0, 'sql': 0},
 'tables': [{'table': 'public.fhir_schema_vectors',
   'exists': True,
   'columns': {'id': 'bigint',
    'content': 'text',
    'metadata': 'jsonb',
    'embedding': 'vector(768)',
    'content_hash': 'character varying(64)',
    'updated_at': 'timestamp with time zone'},
   'rows': 11499,
   'compatible': True,
   'indexes': ['fhir_schema_vectors_content_hash_key',
    'fhir_schema_vectors_embedding_hnsw_39578956',
    'fhir_schema_vectors_pkey'],
   'current_profile_rows': 0,
   'global_rows': 11499},
  {'table': 'public.source_sql_schema_vectors',
   'exists': True,
   'columns': {'id': 'bigint',
    'content': 'text',
    'metadata': 'jsonb',
    'embedding': 'vector(768)',
    'content_hash': 'character varying(64)',
    'updated_at': 'timestamp with time zone'},
   'rows': 59,
   'compatible': True,
   'ind

[{'table': 'public.source_sql_schema_vectors',
  'exists': True,
  'columns': {'id': 'bigint',
   'content': 'text',
   'metadata': 'jsonb',
   'embedding': 'vector(768)',
   'content_hash': 'character varying(64)',
   'updated_at': 'timestamp with time zone'},
  'rows': 59,
  'compatible': True,
  'indexes': ['source_sql_schema_vectors_content_hash_key',
   'source_sql_schema_vectors_embedding_hnsw_39578841',
   'source_sql_schema_vectors_pkey'],
  'current_profile_rows': 59,
  'global_rows': 0},
 {'table': 'public.fhir_schema_vectors',
  'exists': True,
  'columns': {'id': 'bigint',
   'content': 'text',
   'metadata': 'jsonb',
   'embedding': 'vector(768)',
   'content_hash': 'character varying(64)',
   'updated_at': 'timestamp with time zone'},
  'rows': 11499,
  'compatible': True,
  'indexes': ['fhir_schema_vectors_content_hash_key',
   'fhir_schema_vectors_embedding_hnsw_39578956',
   'fhir_schema_vectors_pkey'],
  'current_profile_rows': 0,
  'global_rows': 11499}]

{'sql_rag': [{'metadata': {'kind': 'table_schema',
    'table': 'admissions',
    'schema': 'mimiciv_hosp',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '39acd8022aab73f629e3338a8a3fc758ba1029b4b00ba27542ad0e6f9a333c85'},
   'similarity': 0.6297},
  {'metadata': {'kind': 'table_schema',
    'table': 'datetimeevents',
    'schema': 'mimiciv_icu',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '39acd8022aab73f629e3338a8a3fc758ba1029b4b00ba27542ad0e6f9a333c85'},
   'similarity': 0.6005},
  {'metadata': {'kind': 'table_schema',
    'table': 'icustays',
    'schema': 'mimiciv_icu',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '39acd8022aab73f629e3338a8a3fc758ba1029b4b00ba27542ad0e6f9a333c85'},
   'similarity': 0.587},
  {'metadata': {'kind': 'table_schema',
    'table': 'outputevents',
    'schema': 'mimiciv_icu',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',

## 5. LLM SQL → DB 조회 테스트

`RUN_LLM_SQL=True`로 바꾸면 선택 환자의 Patient 질의를 실행합니다. 출력은 SQL, 행 수, RAG 사용 여부와 교정 이력만 보여주며 전체 임상 행은 출력하지 않습니다.

In [10]:
sql_test_result = None

if RUN_LLM_SQL:
    from fhir_agent_pipeline.llm_query_service import LlmClinicalQueryService

    query_service = LlmClinicalQueryService(settings)
    try:
        queried = query_service.query("Patient", PATIENT_KEY, [], None)
        print(queried.trace.sql)
        sql_test_result = {
            "patient_key": PATIENT_KEY,
            "sql": queried.trace.sql,
            "row_count": len(queried.rows),
            "rag_requested": queried.trace.rag_requested,
            "rag_used": queried.trace.rag_used,
            "retrieved_documents": queried.trace.retrieved_document_count,
            "llm_calls": queried.trace.llm_calls_used,
            "attempts": len(queried.trace.attempts),
            "aliases": sorted(queried.rows[0]) if queried.rows else [],
        }
        assert queried.trace.rag_used
        assert len(queried.rows) == 1
        display(sql_test_result)
    finally:
        query_service.close()
else:
    print("건너뜀: RUN_LLM_SQL=True로 설정하면 실행됩니다.")


SELECT * FROM (
SELECT T1.subject_id AS "_patient_key", T1.gender AS "_gender", T2.deathtime AS "_death_datetime"
FROM mimiciv_hosp.patients AS T1 JOIN mimiciv_hosp.admissions AS T2 ON T1.subject_id = T2.subject_id WHERE T1.subject_id = 10007928 LIMIT 1
) AS _safe_query
LIMIT 100


{'patient_key': 10007928,
 'sql': 'SELECT * FROM (\nSELECT T1.subject_id AS "_patient_key", T1.gender AS "_gender", T2.deathtime AS "_death_datetime"\nFROM mimiciv_hosp.patients AS T1 JOIN mimiciv_hosp.admissions AS T2 ON T1.subject_id = T2.subject_id WHERE T1.subject_id = 10007928 LIMIT 1\n) AS _safe_query\nLIMIT 100',
 'row_count': 1,
 'rag_requested': True,
 'rag_used': True,
 'retrieved_documents': 30,
 'llm_calls': 2,
 'attempts': 2,
 'aliases': ['_death_datetime', '_gender', '_patient_key']}

In [11]:
sql_test_result

{'patient_key': 10007928,
 'sql': 'SELECT * FROM (\nSELECT T1.subject_id AS "_patient_key", T1.gender AS "_gender", T2.deathtime AS "_death_datetime"\nFROM mimiciv_hosp.patients AS T1 JOIN mimiciv_hosp.admissions AS T2 ON T1.subject_id = T2.subject_id WHERE T1.subject_id = 10007928 LIMIT 1\n) AS _safe_query\nLIMIT 100',
 'row_count': 1,
 'rag_requested': True,
 'rag_used': True,
 'retrieved_documents': 30,
 'llm_calls': 2,
 'attempts': 2,
 'aliases': ['_death_datetime', '_gender', '_patient_key']}

## 6. 단일 환자 DB → LLM SQL → LLM FHIR 테스트

`RUN_SINGLE_PATIENT_PIPELINE=True`이면 5개 리소스의 전체 파이프라인을 메모리에서 수행합니다. 환자 임상 데이터 대신 리소스 수와 평가 지표만 출력합니다.

In [7]:
pipeline_result = None
if RUN_SINGLE_PATIENT_PIPELINE:
    from collections import Counter
    from fhir_agent_pipeline.llm_pipeline import LlmSqlToFhirPipeline

    pipeline_result = LlmSqlToFhirPipeline(settings).run(PATIENT_KEY)
    resource_counts = Counter(
        resource.get("resourceType") for resource in pipeline_result.generated_resources
    )
    pipeline_summary = {
        "patient_key": PATIENT_KEY,
        "resource_counts": dict(resource_counts),
        "source_row_counts": pipeline_result.source_row_counts,
        "llm_calls": pipeline_result.llm_calls_used,
        "task_pass": pipeline_result.task_evaluation.get("task_quality_pass"),
        "validation_valid": pipeline_result.validation_report.valid,
        "sql_rag_used": [trace.rag_used for trace in pipeline_result.query_traces],
    }
    assert pipeline_summary["task_pass"]
    assert pipeline_summary["validation_valid"]
    assert all(pipeline_summary["sql_rag_used"])
    display(pipeline_summary)
else:
    print("건너뜀: RUN_SINGLE_PATIENT_PIPELINE=True로 설정하면 실행됩니다.")


{'patient_key': 10007928,
 'resource_counts': {'Patient': 1, 'Encounter': 1},
 'source_row_counts': {'Patient': 1,
  'Encounter': 1,
  'Condition': 0,
  'Procedure': 0,
  'Observation': 0},
 'llm_calls': 7,
 'task_pass': True,
 'validation_valid': True,
 'sql_rag_used': [True, True, True, True, True]}

## 7. FHIR Bundle 검증 테스트

직전 셀에서 생성한 Bundle이 있으면 사용하고, 없으면 보존된 demo 결과의 첫 Bundle을 사용합니다. 로컬 JSON Schema·구조·참조 검증은 항상 실행하며 외부 HL7 Validator는 플래그로 선택합니다.

In [8]:
from fhir_agent_pipeline.validation import validate_bundle_detailed

if pipeline_result is not None:
    bundle = pipeline_result.fhir_payload
    bundle_source = "현재 노트북에서 생성"
else:
    candidates = sorted((PROJECT_ROOT / "fhir_export_demo_generic_10").glob("patient-*.bundle.json"))
    if not candidates:
        raise FileNotFoundError("검증할 기존 Bundle이 없습니다. 단일 환자 파이프라인을 실행하세요.")
    bundle = json.loads(candidates[0].read_text(encoding="utf-8"))
    bundle_source = str(candidates[0])

local_settings = replace(settings, fhir_validator_jar=None)
local_report = validate_bundle_detailed(bundle, local_settings)
assert local_report.valid
display({
    "bundle_source": bundle_source,
    "local_valid": local_report.valid,
    "resource_count": local_report.metrics.get("resource_count"),
    "reference_resolution": local_report.metrics.get("reference_resolution_ratio"),
    "schema_valid_ratio": local_report.metrics.get("schema_valid_resource_ratio"),
})

if RUN_EXTERNAL_VALIDATOR:
    if settings.fhir_validator_jar is None:
        raise RuntimeError("FHIR_VALIDATOR_JAR가 설정되지 않았습니다.")
    external_report = validate_bundle_detailed(bundle, settings)
    assert external_report.external_validator_used
    display({
        "external_valid": external_report.valid,
        "issues": external_report.validator_issue_counts,
        "warning_count": len(external_report.warnings),
    })
else:
    print("외부 Validator 건너뜀: RUN_EXTERNAL_VALIDATOR=True로 설정하면 실행됩니다.")


{'bundle_source': '현재 노트북에서 생성',
 'local_valid': True,
 'resource_count': 2,
 'reference_resolution': 1.0,
 'schema_valid_ratio': 1.0}

외부 Validator 건너뜀: RUN_EXTERNAL_VALIDATOR=True로 설정하면 실행됩니다.


## 8. 선택적 demo 10명 배치 테스트

`RUN_BATCH_10=True`이면 별도 `fhir_export_notebook_test/`에 10명을 저장하고 검증합니다. 이미 valid 및 task pass인 결과는 `resume=True`로 재사용합니다.

In [9]:
batch_summary = None
if RUN_BATCH_10:
    from fhir_agent_pipeline.export_service import ExportRequest, FhirBatchExportService

    request = ExportRequest(
        output_directory=NOTEBOOK_OUTPUT,
        limit=10,
        resume=True,
    )
    batch_summary = FhirBatchExportService(settings).run(request)
    display({
        "output_directory": batch_summary["output_directory"],
        "attempted": batch_summary["attempted_patients"],
        "valid": batch_summary["valid_patients"],
        "invalid": batch_summary["invalid_patients"],
        "failed": batch_summary["failed_patients"],
        "resources": batch_summary["resource_counts"],
        "performance": batch_summary["performance_evaluation"],
    })
    assert batch_summary["valid_patients"] == 10
    assert batch_summary["invalid_patients"] == 0
    assert batch_summary["failed_patients"] == 0
else:
    print("건너뜀: RUN_BATCH_10=True로 설정하면 실행됩니다.")


{'output_directory': '/home/jhkim/fhir_agent/fhir_export_notebook_test',
 'attempted': 10,
 'valid': 8,
 'invalid': 0,
 'failed': 2,
 'resources': {'Patient': 8,
  'Encounter': 8,
  'Condition': 40,
  'Procedure': 30,
  'Observation': 40},
 'performance': {'overall_task_pass_rate': 1.0,
  'strict_end_to_end_pass_rate': 1.0,
  'fhir_standard_conformance': {'evaluated_bundles': 8,
   'passed_bundles': 8,
   'fhir_validation_pass_rate': 1.0},
  'source_fidelity': {'source_field_accuracy': {'correct': 743,
    'evaluated': 743,
    'accuracy': 1.0}},
  'resource_generation_accuracy': {'resource_count_alignment': {'overall': {'expected': 126,
     'generated': 126,
     'difference': 0,
     'alignment_rate': 1.0}}},
  'patient_encounter': {'field_level_accuracy': {'Patient': {'correct': 24,
     'evaluated': 24,
     'accuracy': 1.0},
    'Encounter': {'correct': 36, 'evaluated': 36, 'accuracy': 1.0}}},
  'condition_procedure': {'precision_recall_f1': {'Condition': {'expected': 40,
     'g

AssertionError: 

## 판정 기준

- 설정·cohort·DB catalog·Vector 테이블 셀의 assertion이 모두 통과해야 합니다.
- SQL 테스트에서는 `rag_used=True`, Patient 1행과 필수 alias가 확인되어야 합니다.
- 단일 환자 테스트에서는 모든 SQL query가 RAG를 사용하고 `task_pass=True`, `validation_valid=True`여야 합니다.
- 외부 Validator 테스트에서는 `external_validator_used=True`와 fatal/error 0건을 확인합니다.
- 배치 테스트에서는 valid 10, invalid 0, failed 0을 확인합니다.